# ENSF617 Main Training Notebook

This notebook uses the shared helpers in `main.py` to prepare data, train the model, run test evaluation, and optionally save predictions.

**How to use it:** edit the configuration cell, run the config-building cell, then run the workflow cell.

**Disclaimer:** this notebook is intended for research and development use inside the repository. The default configuration is a baseline starting point, not a claim of optimal performance or clinical validity.

In [ ]:
# Notebook bootstrap:
# - assume the notebook is launched from the repository root
# - make the repo root importable so `defaults.py` and `main.py` can be imported
# - reuse the same shared helpers as the CLI entrypoint
from pathlib import Path
import sys

ROOT_DIR = Path.cwd()
assert (ROOT_DIR / "main.py").exists(), "Run this notebook from the repository root."
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from defaults import (
    DEFAULT_AZT1D_URL,
    build_default_config,
    build_default_snapshot_config,
    build_default_train_config,
)
from main import (
    run_training_workflow,
)


In [ ]:
# Edit these values for your run.
#
# These are intentionally plain notebook variables rather than a single
# nested dict so it stays easy to tweak one thing at a time while
# experimenting.
output_dir = ROOT_DIR / "artifacts" / "notebook_run"
dataset_url = DEFAULT_AZT1D_URL
processed_dir = ROOT_DIR / "data" / "processed"
processed_file_name = "azt1d_processed.csv"

encoder_length = 168
prediction_length = 12
batch_size = 64
num_workers = 0

max_epochs = 20
accelerator = "auto"
devices = "auto"
precision = 32

learning_rate = 1e-3
weight_decay = 0.0
optimizer_name = "adam"
seed = 42

skip_test = False
skip_predict = False
save_predictions = True


In [ ]:
# Convert the editable notebook variables into the typed config objects
# used by the rest of the codebase.
#
# Keeping this step explicit makes it easier to inspect what exact config
# will be handed to the training workflow before launching a longer run.
config = build_default_config(
    dataset_url=dataset_url,
    raw_dir=ROOT_DIR / "data" / "raw",
    cache_dir=ROOT_DIR / "data" / "cache",
    extracted_dir=ROOT_DIR / "data" / "extracted",
    processed_dir=processed_dir,
    processed_file_name=processed_file_name,
    encoder_length=encoder_length,
    prediction_length=prediction_length,
    batch_size=batch_size,
    num_workers=num_workers,
)

train_config = build_default_train_config(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=devices,
    precision=precision,
    default_root_dir=output_dir,
)

snapshot_config = build_default_snapshot_config(output_dir=output_dir)

# Display the top-level config so the notebook user can sanity-check it.
config


In [ ]:
# Run the shared end-to-end workflow.
#
# This cell is intentionally thin: it calls the same Python function used
# by `main.py`, which helps keep notebook and script behavior aligned.
artifacts = run_training_workflow(
    config,
    train_config=train_config,
    snapshot_config=snapshot_config,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    optimizer_name=optimizer_name,
    output_dir=output_dir,
    seed=seed,
    skip_test=skip_test,
    skip_predict=skip_predict,
    save_predictions=save_predictions,
)


# Show the compact JSON-friendly summary written for this run.
artifacts.summary


In [ ]:
# Held-out test metrics, if a test split was available and `skip_test`
# was left as False.
artifacts.test_metrics

In [ ]:
# Inspect one raw prediction batch.
#
# The saved tensors are left in their direct model-output form so later
# notebook cells can decide how to aggregate, visualize, or calibrate
# them.
if artifacts.test_predictions:
    first_batch = artifacts.test_predictions[0]
    print(first_batch.shape)
    first_batch
else:
    print("No test predictions were produced for this run.")
